In [1]:
!pip uninstall -y torchao -q
!pip install -q torchao>=0.16.0
!pip install -q --upgrade transformers datasets peft accelerate bitsandbytes

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
from datasets import Dataset

data = [
    {"text": "### Question: What is an Operating System?\n### Answer:\nDefinition: Software that manages hardware and software resources\nKey Points:\n- Interface between user and hardware\n- Manages processes and memory\nExample: Windows, Linux"},

    {"text": "### Question: What is Deadlock?\n### Answer:\nDefinition: Situation where processes wait indefinitely\nKey Points:\n- Caused by resource contention\n- No process proceeds\nExample: Circular wait"},

    {"text": "### Question: What is Database?\n### Answer:\nDefinition: Organized collection of data\nKey Points:\n- Enables storage and retrieval\n- Structured format\nExample: MySQL"},

    {"text": "### Question: What is SQL?\n### Answer:\nDefinition: Language to manage databases\nKey Points:\n- Used for queries\n- Supports CRUD operations\nExample: SELECT * FROM table"},

    {"text": "### Question: What is Normalization?\n### Answer:\nDefinition: Process to organize database tables\nKey Points:\n- Reduces redundancy\n- Improves consistency\nExample: 1NF, 2NF"},
]

# 🔥 repeat for stronger learning
data = data * 6

dataset = Dataset.from_list(data)

In [4]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [5]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [6]:
def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=80)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [12]:
test_prompt = "### Question: What is Database?\n### Answer:\nDefinition:"
before_output = generate(model, test_prompt)

print("🔴 BEFORE FINE-TUNING:\n")
print(before_output)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔴 BEFORE FINE-TUNING:

### Question: What is Database?
### Answer:
Definition: a collection of data organized in a structured way for easy retrieval and manipulation



In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=10,   # 🔥 important
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

torch.cuda.empty_cache()

train_result = trainer.train()

Step,Training Loss
1,0.674171
2,0.782037
3,0.656258
4,0.638657
5,0.732413
6,0.712330
7,0.575945
8,0.669063
9,0.617624
10,0.594708


In [14]:
print("\n📉 TRAINING METRICS:\n")
print(train_result.metrics)


📉 TRAINING METRICS:

{'train_runtime': 68.7983, 'train_samples_per_second': 4.361, 'train_steps_per_second': 2.18, 'total_flos': 238611175833600.0, 'train_loss': 0.2782067840298017, 'epoch': 10.0}


In [15]:
after_output = generate(model, test_prompt)

print("\n🟢 AFTER FINE-TUNING:\n")
print(after_output)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🟢 AFTER FINE-TUNING:

### Question: What is Database?
### Answer:
Definition: Organized collection of data
Key Points:
- Enables storage and retrieval
- Structured format
Example: MySQL


In [16]:
print(f"""
================= FINAL DEMO =================

🔴 BEFORE:
{before_output}

🟢 AFTER:
{after_output}

📉 FINAL TRAIN LOSS:
{train_result.metrics['train_loss']}

==============================================
""")


================= FINAL DEMO =================

🔴 BEFORE:
### Question: What is Database?
### Answer:
Definition: a collection of data organized in a structured way for easy retrieval and manipulation


🟢 AFTER:
### Question: What is Database?
### Answer:
Definition: Organized collection of data
Key Points:
- Enables storage and retrieval
- Structured format
Example: MySQL

📉 FINAL TRAIN LOSS:
0.2782067840298017


